In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects = 1000
num_features = 10

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0,    # X5 effect
    0.0,
    0.0,
    0.0,
    0.0,
    0.0,
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 1

#Y_raw = X @ beta_true + noise  + X[:,4:5]*X[:,4:5] +X[:,2:3]*X[:,3:4]
Y_raw = X @ beta_true + noise  +X[:,2:3]*X[:,3:4]+ X[:,4:5]*X[:,4:5]

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 1.9105],
        [ 2.0406],
        [-1.3298],
        [ 0.6045],
        [ 0.0702],
        [ 2.9384],
        [-0.0119],
        [-0.1319],
        [ 0.0841],
        [-0.1123],
        [-0.0087]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([12, 12])

Attention Matrix:
 tensor([[0.0949, 0.0788, 0.0697, 0.0484, 0.1196, 0.0566, 0.1037, 0.0656, 0.0587,
         0.0951, 0.1786, 0.0302],
        [0.1151, 0.0846, 0.1101, 0.0942, 0.1201, 0.0632, 0.0866, 0.0670, 0.0826,
         0.0568, 0.0681, 0.0515],
        [0.0566, 0.0721, 0.0495, 0.0269, 0.0258, 0.0368, 0.0270, 0.0234, 0.0334,
         0.0519, 0.0178, 0.5788],
        [0.0952, 0.0400, 0.0626, 0.0966, 0.0659, 0.1297, 0.1034, 0.0729, 0.0800,
         0.0435, 0.1388, 0.0715],
        [0.0580, 0.0738, 0.1090, 0.1201, 0.0604, 0.0847, 0.0910, 0.0862, 0.1145,
         0.0538, 0.1008, 0.0476],
        [0.0903, 0.0876, 0.0648, 0.0477, 0.0468, 0.0868, 0.0627, 0.0584, 0.0922,
         0.1274, 0.0724, 0.1629],
        [0.0465, 0.0487, 0.0190, 0.0275, 0.0477, 0.0627, 0.0328, 0.0342, 0.0213,
         0.0317, 0.0472, 0.5807],
        [0.0574, 0.0744, 0.0736, 0.1124, 0.0484, 0.0868, 0.0810, 0.0823, 0.1365,
         0.1533, 0.0483, 0.0455],
        [0.0472

In [8]:
# 把Y再從矩陣中拿掉
A = attention_matrix[:-1,:-1]

print("Attention Matrix Shape(拿掉Y):", A.shape)
print("\nAttention Matrix(拿掉Y):\n", A)

Attention Matrix Shape(拿掉Y): torch.Size([11, 11])

Attention Matrix(拿掉Y):
 tensor([[0.0949, 0.0788, 0.0697, 0.0484, 0.1196, 0.0566, 0.1037, 0.0656, 0.0587,
         0.0951, 0.1786],
        [0.1151, 0.0846, 0.1101, 0.0942, 0.1201, 0.0632, 0.0866, 0.0670, 0.0826,
         0.0568, 0.0681],
        [0.0566, 0.0721, 0.0495, 0.0269, 0.0258, 0.0368, 0.0270, 0.0234, 0.0334,
         0.0519, 0.0178],
        [0.0952, 0.0400, 0.0626, 0.0966, 0.0659, 0.1297, 0.1034, 0.0729, 0.0800,
         0.0435, 0.1388],
        [0.0580, 0.0738, 0.1090, 0.1201, 0.0604, 0.0847, 0.0910, 0.0862, 0.1145,
         0.0538, 0.1008],
        [0.0903, 0.0876, 0.0648, 0.0477, 0.0468, 0.0868, 0.0627, 0.0584, 0.0922,
         0.1274, 0.0724],
        [0.0465, 0.0487, 0.0190, 0.0275, 0.0477, 0.0627, 0.0328, 0.0342, 0.0213,
         0.0317, 0.0472],
        [0.0574, 0.0744, 0.0736, 0.1124, 0.0484, 0.0868, 0.0810, 0.0823, 0.1365,
         0.1533, 0.0483],
        [0.0472, 0.0504, 0.0783, 0.1157, 0.0532, 0.1269, 0.0980, 0.11

In [9]:
# 矩陣乘上Y的變異數
var_y = torch.var(Y_train)

A_var_y = A * var_y

In [10]:
A_var_y

tensor([[1.5758, 1.3091, 1.1575, 0.8043, 1.9856, 0.9398, 1.7227, 1.0891, 0.9740,
         1.5798, 2.9660],
        [1.9120, 1.4045, 1.8283, 1.5648, 1.9940, 1.0494, 1.4389, 1.1132, 1.3713,
         0.9437, 1.1310],
        [0.9403, 1.1968, 0.8218, 0.4475, 0.4290, 0.6105, 0.4480, 0.3884, 0.5543,
         0.8620, 0.2958],
        [1.5807, 0.6636, 1.0393, 1.6046, 1.0941, 2.1535, 1.7168, 1.2107, 1.3285,
         0.7224, 2.3047],
        [0.9627, 1.2252, 1.8103, 1.9941, 1.0032, 1.4063, 1.5119, 1.4321, 1.9021,
         0.8931, 1.6741],
        [1.4992, 1.4545, 1.0768, 0.7923, 0.7778, 1.4408, 1.0412, 0.9695, 1.5308,
         2.1149, 1.2024],
        [0.7728, 0.8091, 0.3149, 0.4566, 0.7925, 1.0420, 0.5439, 0.5671, 0.3530,
         0.5266, 0.7838],
        [0.9533, 1.2350, 1.2223, 1.8665, 0.8037, 1.4420, 1.3444, 1.3672, 2.2668,
         2.5462, 0.8022],
        [0.7836, 0.8375, 1.2999, 1.9205, 0.8836, 2.1076, 1.6280, 1.8675, 2.2367,
         1.0321, 1.7151],
        [1.0495, 0.8487, 1.2847, 2.77

In [11]:
X_train.T @ X_train

tensor([[800.0000,  11.7720,  17.2216,  49.5932,  -6.4263, -32.1524, -20.6207,
          -4.8550,  55.3996,  -3.8815,   7.4138],
        [ 11.7720, 785.9890,  69.9708,  24.3736, -18.1999, -52.4016, -15.7686,
         -16.8365, -12.5653,  11.5560,  11.5038],
        [ 17.2216,  69.9708, 790.1130, -49.6005,  15.4388,  -4.3037, -35.9303,
          25.3849, -55.9114,  11.0519, -22.8567],
        [ 49.5932,  24.3736, -49.6005, 777.2329,  36.4230,   5.6587, -25.0208,
          59.6404,  19.3818, -35.0106, -17.9095],
        [ -6.4263, -18.1999,  15.4388,  36.4230, 763.5020, -19.3683,   8.8376,
          12.9132,  -3.4125,  -4.1807,  35.1357],
        [-32.1524, -52.4016,  -4.3037,   5.6587, -19.3683, 749.6162,   0.9192,
          45.5000,  36.0127, -21.4020,  10.5270],
        [-20.6207, -15.7686, -35.9303, -25.0208,   8.8376,   0.9192, 804.2502,
         -14.1605,  -5.1981,   1.2920, -11.1902],
        [ -4.8550, -16.8365,  25.3849,  59.6404,  12.9132,  45.5000, -14.1605,
         759.3004,

In [12]:

if torch.linalg.det(A)<0:
    beta_attention = torch.linalg.solve(X_train.T @ X_train + A*torch.trace(A), X_train.T @ Y_train)
else:
    beta_attention = torch.linalg.solve(X_train.T @ X_train - A*torch.trace(A), X_train.T @ Y_train)



In [13]:
print(torch.linalg.det(A))
print(torch.trace(A))
print(torch.trace(X_train.T @ X_train))

tensor(-7.3107e-15, grad_fn=<LinalgDetBackward0>)
tensor(0.8563, grad_fn=<TraceBackward0>)
tensor(8612.7910)


In [14]:
torch.linalg.det(A)

tensor(-7.3107e-15, grad_fn=<LinalgDetBackward0>)

In [15]:
torch.linalg.det(A.T@A)

tensor(5.3408e-29, grad_fn=<LinalgDetBackward0>)

In [16]:
torch.linalg.det(X_train.T @ X_train)

tensor(6.2986e+31)

In [17]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

# 算MSE
mse_attention = torch.mean(
    (Y_test - Y_pred_attention)**2
)

In [18]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

print(
    "OLS beta:",
    beta_ols
)

Y_pred_ols = X_test @ beta_ols

mse_ols = torch.mean(
    (Y_test - Y_pred_ols)**2
)



OLS beta: tensor([[ 1.9053],
        [ 2.0258],
        [-1.2698],
        [ 0.5813],
        [ 0.1068],
        [ 2.9532],
        [ 0.0141],
        [-0.0668],
        [ 0.0598],
        [-0.1617],
        [ 0.0397]])


In [19]:
print(
    "Attention MSE:",
    mse_attention.item()
)

print()

print(
    "OLS MSE:      ",
    mse_ols.item()
)

Attention MSE: 4.612683296203613

OLS MSE:       4.613204479217529
